<a href="https://colab.research.google.com/github/pngl2415/2.6-openMP/blob/main/Actividad_3_6_Problema_pr%C3%A1ctico_en_CUDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!nvidia-smi
!nvcc --version


Sat Nov 14 11:27:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

%%writefile busqueda_fuerza_bruta.cu
/*
 ACTIVIDAD 3.6 - BUSQUEDA POR FUERZA BRUTA CON CUDA

 INTEGRANTES (ORDEN ALFABETICO):
 1. Guzman Lopez Paola Nicole
 2. Mondragon Gambino Juan Sinuhe
 3. Morales Limon Jennifer Nataly

 OBJETIVO:
 Implementar una busqueda paralela de claves de prueba
 usando CUDA, distribuyendo el espacio de busqueda
 entre bloques e hilos de la GPU.
*/

#include <cuda_runtime.h>
#include <iostream>
#include <iomanip>
#include <string>
#include <sstream>
#include <chrono>
#include <cstdint>
#include <algorithm>

using namespace std;

const int BLOQUES = 1000;
const int HILOS_POR_BLOQUE = 1024;
const int LONGITUD_MAXIMA = 10;
const int BASE = 36;

const unsigned long long TAM_LOTE = 8192000ULL;
const unsigned long long LIMITE_PREDETERMINADO = 8192000ULL;
const unsigned long long LIMITE_MAXIMO = 50000000ULL;

const char ALFABETO[] = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789";

const string INTEGRANTES =
    "Guzman Lopez Paola Nicole | "
    "Mondragon Gambino Juan Sinuhe | "
    "Morales Limon Jennifer Nataly";

struct Objetivo {
    char clave[11];
    int longitud;
};

struct Resultado {
    bool encontrado;
    unsigned long long intentosAsignados;
    unsigned long long indice;
    double tiempoGPU;
    double tiempoTotal;
    int lotes;
    bool error;
    string mensaje;
};

bool verificarCUDA(cudaError_t error, const char* operacion) {
    if (error != cudaSuccess) {
        cerr << "[ERROR CUDA] " << operacion << ": "
             << cudaGetErrorString(error) << endl;
        return false;
    }
    return true;
}

void separador() {
    cout << "============================================================\n";
}

void titulo(const string& texto) {
    cout << "\n";
    separador();
    cout << " " << texto << "\n";
    separador();
}

unsigned long long potencia(unsigned long long base, int exponente) {
    unsigned long long resultado = 1;
    for (int i = 0; i < exponente; i++) {
        resultado *= base;
    }
    return resultado;
}

bool convertirEntero(const string& texto, unsigned long long& numero) {
    if (texto.empty()) return false;

    for (char c : texto) {
        if (c < '0' || c > '9') return false;
    }

    try {
        size_t posicion = 0;
        numero = stoull(texto, &posicion);
        return posicion == texto.size();
    } catch (...) {
        return false;
    }
}

unsigned long long pedirNumero(
    const string& mensaje,
    unsigned long long minimo,
    unsigned long long maximo
) {
    string entrada;
    unsigned long long numero = 0;

    while (true) {
        cout << mensaje;

        if (!getline(cin, entrada)) {
            cout << "\n[INFO] Entrada finalizada.\n";
            return minimo;
        }

        if (convertirEntero(entrada, numero) &&
            numero >= minimo && numero <= maximo) {
            return numero;
        }

        cout << "[ERROR] Introduce un numero entre "
             << minimo << " y " << maximo << ".\n";
    }
}

bool validarClave(const string& clave, int minimo, int maximo) {
    if (clave.size() < static_cast<size_t>(minimo) ||
        clave.size() > static_cast<size_t>(maximo)) {
        return false;
    }

    for (char c : clave) {
        if (!((c >= 'A' && c <= 'Z') ||
              (c >= '0' && c <= '9'))) {
            return false;
        }
    }

    return true;
}

// Convierte un indice numerico en una clave alfanumerica.
__host__ __device__
void indiceAClave(
    unsigned long long indice,
    int longitud,
    char* clave
) {
    const char caracteres[] = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789";

    for (int i = longitud - 1; i >= 0; i--) {
        clave[i] = caracteres[indice % BASE];
        indice /= BASE;
    }

    clave[longitud] = '\0';
}

// Compara la clave generada con la clave objetivo.
__device__
bool verificarClave(
    const char* generada,
    const Objetivo& objetivo
) {
    for (int i = 0; i < objetivo.longitud; i++) {
        if (generada[i] != objetivo.clave[i]) {
            return false;
        }
    }
    return true;
}

/*
 KERNEL CUDA

 Cada hilo recibe un rango diferente:
 [inicioHilo, finHilo).

 La bandera 'encontrado' permite avisar a los demas
 hilos que ya se localizo la clave.

 El proceso anfitrion deja de lanzar nuevos lotes
 cuando detecta la bandera.
*/
__global__
void kernelFuerzaBruta(
    unsigned long long inicioLote,
    unsigned long long finLote,
    Objetivo objetivo,
    int* encontrado,
    unsigned long long* indiceEncontrado
) {
    unsigned long long id =
        static_cast<unsigned long long>(blockIdx.x) *
        blockDim.x + threadIdx.x;

    unsigned long long totalHilos =
        static_cast<unsigned long long>(gridDim.x) *
        blockDim.x;

    unsigned long long cantidad = finLote - inicioLote;

    unsigned long long porHilo =
        cantidad / totalHilos +
        (cantidad % totalHilos != 0ULL ? 1ULL : 0ULL);

    unsigned long long desplazamiento = id * porHilo;

    if (desplazamiento >= cantidad) return;

    unsigned long long inicioHilo =
        inicioLote + desplazamiento;

    unsigned long long finHilo =
        inicioHilo + porHilo;

    if (finHilo > finLote) {
        finHilo = finLote;
    }

    char candidata[11];

    for (unsigned long long indice = inicioHilo;
         indice < finHilo;
         indice++) {

        if (atomicAdd(encontrado, 0) != 0) {
            break;
        }

        indiceAClave(indice, objetivo.longitud, candidata);

        if (verificarClave(candidata, objetivo)) {
            if (atomicCAS(encontrado, 0, 1) == 0) {
                *indiceEncontrado = indice;
            }
            break;
        }
    }
}

bool claveAIndice(
    const string& clave,
    unsigned long long& indice
) {
    if (!validarClave(clave, 1, LONGITUD_MAXIMA))
        return false;

    indice = 0;

    for (char c : clave) {
        int valor = -1;

        for (int i = 0; i < BASE; i++) {
            if (ALFABETO[i] == c) {
                valor = i;
                break;
            }
        }

        if (valor < 0) return false;

        indice = indice * BASE + valor;
    }

    return true;
}

Resultado ejecutarBusqueda(
    const string& clave,
    unsigned long long inicio,
    unsigned long long presupuesto
) {
    Resultado resultado = {};
    resultado.indice = 0;
    resultado.mensaje = "Sin ejecutar";

    int longitud = static_cast<int>(clave.size());

    if (!validarClave(clave, 1, LONGITUD_MAXIMA)) {
        resultado.error = true;
        resultado.mensaje = "Clave invalida";
        return resultado;
    }

    unsigned long long espacio = potencia(BASE, longitud);

    if (inicio >= espacio || presupuesto == 0) {
        resultado.error = true;
        resultado.mensaje = "Rango de busqueda invalido";
        return resultado;
    }

    int dispositivos = 0;

    if (!verificarCUDA(
            cudaGetDeviceCount(&dispositivos),
            "Consultar dispositivos")) {
        resultado.error = true;
        resultado.mensaje = "No se pudo consultar CUDA";
        return resultado;
    }

    if (dispositivos == 0) {
        resultado.error = true;
        resultado.mensaje = "No hay una GPU CUDA disponible";
        return resultado;
    }

    cudaDeviceProp propiedades;

    if (!verificarCUDA(
            cudaGetDeviceProperties(&propiedades, 0),
            "Consultar GPU")) {
        resultado.error = true;
        resultado.mensaje = "No se pudieron leer propiedades GPU";
        return resultado;
    }

    if (propiedades.maxThreadsPerBlock < HILOS_POR_BLOQUE) {
        resultado.error = true;
        resultado.mensaje = "La GPU no admite 1024 hilos por bloque";
        return resultado;
    }

    unsigned long long cantidad =
        min(presupuesto, espacio - inicio);

    unsigned long long fin = inicio + cantidad;

    Objetivo objetivo = {};
    objetivo.longitud = longitud;

    for (int i = 0; i < longitud; i++) {
        objetivo.clave[i] = clave[i];
    }

    objetivo.clave[longitud] = '\0';

    int* d_encontrado = nullptr;
    unsigned long long* d_indice = nullptr;

    cudaEvent_t eventoInicio = nullptr;
    cudaEvent_t eventoFin = nullptr;

    auto inicioTotal = chrono::steady_clock::now();

    auto liberar = [&]() {
        if (eventoInicio) cudaEventDestroy(eventoInicio);
        if (eventoFin) cudaEventDestroy(eventoFin);
        if (d_encontrado) cudaFree(d_encontrado);
        if (d_indice) cudaFree(d_indice);
    };

    if (!verificarCUDA(
            cudaMalloc((void**)&d_encontrado, sizeof(int)),
            "Reservar bandera")) {
        resultado.error = true;
        resultado.mensaje = "Error al reservar memoria";
        return resultado;
    }

    if (!verificarCUDA(
            cudaMalloc((void**)&d_indice, sizeof(unsigned long long)),
            "Reservar indice")) {
        liberar();
        resultado.error = true;
        resultado.mensaje = "Error al reservar memoria";
        return resultado;
    }

    if (!verificarCUDA(
            cudaMemset(d_encontrado, 0, sizeof(int)),
            "Inicializar bandera") ||
        !verificarCUDA(
            cudaMemset(d_indice, 0, sizeof(unsigned long long)),
            "Inicializar indice") ||
        !verificarCUDA(
            cudaEventCreate(&eventoInicio),
            "Crear evento inicial") ||
        !verificarCUDA(
            cudaEventCreate(&eventoFin),
            "Crear evento final")) {
        liberar();
        resultado.error = true;
        resultado.mensaje = "Error de inicializacion CUDA";
        return resultado;
    }

    titulo("CONFIGURACION DE BUSQUEDA");

    cout << "GPU: " << propiedades.name << "\n";
    cout << "Clave objetivo: " << clave << "\n";
    cout << "Longitud: " << longitud << "\n";
    cout << "Alfabeto: " << ALFABETO << "\n";
    cout << "Espacio total: " << espacio << "\n";
    cout << "Indice inicial: " << inicio << "\n";
    cout << "Indice final exclusivo: " << fin << "\n";
    cout << "Indices asignados: " << cantidad << "\n";
    cout << "Bloques CUDA: " << BLOQUES << "\n";
    cout << "Hilos por bloque: " << HILOS_POR_BLOQUE << "\n";
    cout << "Total de hilos: "
         << BLOQUES * HILOS_POR_BLOQUE << "\n";

    cout << "\n[AVISO] La busqueda puede tardar varios minutos.\n";
    cout << "[INFO] Cada hilo procesa un rango exclusivo.\n";
    cout << "[INFO] La busqueda se detiene al detectar la clave\n";
    cout << "       o al alcanzar el limite de indices asignados.\n";

    int bandera = 0;

    if (!verificarCUDA(
            cudaEventRecord(eventoInicio),
            "Iniciar cronometro GPU")) {
        liberar();
        resultado.error = true;
        resultado.mensaje = "Error de cronometro CUDA";
        return resultado;
    }

    for (unsigned long long actual = inicio;
         actual < fin && bandera == 0;) {

        unsigned long long finActual =
            actual + min(TAM_LOTE, fin - actual);

        unsigned long long cantidadLote = finActual - actual;

        kernelFuerzaBruta<<<BLOQUES, HILOS_POR_BLOQUE>>>(
            actual,
            finActual,
            objetivo,
            d_encontrado,
            d_indice
        );

        if (!verificarCUDA(
                cudaGetLastError(),
                "Lanzamiento del kernel") ||
            !verificarCUDA(
                cudaDeviceSynchronize(),
                "Sincronizar kernel") ||
            !verificarCUDA(
                cudaMemcpy(
                    &bandera,
                    d_encontrado,
                    sizeof(int),
                    cudaMemcpyDeviceToHost
                ),
                "Leer bandera")) {

            resultado.error = true;
            resultado.mensaje = "Error durante busqueda CUDA";
            break;
        }

        resultado.intentosAsignados += cantidadLote;
        resultado.lotes++;

        cout << "[LOTE " << resultado.lotes << "] "
             << "Rango [" << actual << ", " << finActual
             << ") | Estado: "
             << (bandera ? "ENCONTRADA" : "FINALIZADO")
             << "\n";

        actual = finActual;
    }

    if (!resultado.error && bandera != 0) {
        if (!verificarCUDA(
                cudaMemcpy(
                    &resultado.indice,
                    d_indice,
                    sizeof(unsigned long long),
                    cudaMemcpyDeviceToHost
                ),
                "Recuperar indice encontrado")) {
            resultado.error = true;
            resultado.mensaje = "Error al recuperar resultado";
        } else {
            resultado.encontrado = true;
        }
    }

    if (!verificarCUDA(
            cudaEventRecord(eventoFin),
            "Detener cronometro GPU") ||
        !verificarCUDA(
            cudaEventSynchronize(eventoFin),
            "Sincronizar evento final")) {
        resultado.error = true;
        resultado.mensaje = "Error de medicion CUDA";
    } else {
        float milisegundos = 0;

        if (verificarCUDA(
                cudaEventElapsedTime(
                    &milisegundos,
                    eventoInicio,
                    eventoFin
                ),
                "Calcular tiempo GPU")) {
            resultado.tiempoGPU = milisegundos / 1000.0;
        } else {
            resultado.error = true;
            resultado.mensaje = "Error al medir tiempo GPU";
        }
    }

    auto finTotal = chrono::steady_clock::now();

    resultado.tiempoTotal =
        chrono::duration<double>(finTotal - inicioTotal).count();

    if (!resultado.error) {
        resultado.mensaje = resultado.encontrado ?
            "Clave encontrada correctamente" :
            "Clave no encontrada dentro del rango asignado";
    }

    liberar();

    return resultado;
}

void mostrarResultado(
    const string& clave,
    const Resultado& resultado
) {
    titulo("RESULTADO DE LA EJECUCION");

    if (resultado.error) {
        cout << "[ERROR] " << resultado.mensaje << "\n";
        return;
    }

    cout << "Estado: "
         << (resultado.encontrado ? "ENCONTRADA" : "NO ENCONTRADA")
         << "\n";

    cout << "Clave objetivo: " << clave << "\n";

    if (resultado.encontrado) {
        char recuperada[11];
        indiceAClave(
            resultado.indice,
            static_cast<int>(clave.size()),
            recuperada
        );

        cout << "Clave recuperada: " << recuperada << "\n";
        cout << "Indice encontrado: " << resultado.indice << "\n";
    }

    cout << "Indices asignados: "
         << resultado.intentosAsignados << "\n";

    cout << "Lotes ejecutados: " << resultado.lotes << "\n";

    cout << fixed << setprecision(6);
    cout << "Tiempo GPU: " << resultado.tiempoGPU << " segundos\n";
    cout << "Tiempo total: " << resultado.tiempoTotal << " segundos\n";

    cout << "Observacion: " << resultado.mensaje << "\n";

    if (!resultado.encontrado) {
        cout << "[INFO] Esto no demuestra que la clave no exista.\n";
        cout << "       Solo se reviso el rango asignado.\n";
    }

    cout << "[NOTA] Los indices asignados no representan\n";
    cout << "       necesariamente comparaciones realizadas,\n";
    cout << "       porque los hilos pueden detenerse antes.\n";
}

void mostrarGPU() {
    titulo("INFORMACION DE GPU");

    int cantidad = 0;

    if (!verificarCUDA(
            cudaGetDeviceCount(&cantidad),
            "Consultar dispositivos")) return;

    if (cantidad == 0) {
        cout << "[ERROR] No se encontro una GPU CUDA.\n";
        return;
    }

    for (int i = 0; i < cantidad; i++) {
        cudaDeviceProp p;

        if (!verificarCUDA(
                cudaGetDeviceProperties(&p, i),
                "Consultar propiedades GPU")) return;

        cout << "Dispositivo: " << i << "\n";
        cout << "Nombre: " << p.name << "\n";
        cout << "Multiprocesadores: " << p.multiProcessorCount << "\n";
        cout << "Hilos maximos por bloque: "
             << p.maxThreadsPerBlock << "\n";
        cout << "Memoria global (MB): "
             << p.totalGlobalMem / (1024 * 1024) << "\n";
        cout << "CUDA Compute Capability: "
             << p.major << "." << p.minor << "\n";
        cout << "Bloques configurados: " << BLOQUES << "\n";
        cout << "Hilos por bloque: " << HILOS_POR_BLOQUE << "\n";
        cout << "Total de hilos: "
             << BLOQUES * HILOS_POR_BLOQUE << "\n";
    }
}

void mostrarEspacio() {
    titulo("ESPACIO DE BUSQUEDA");

    cout << "Alfabeto: " << ALFABETO << "\n";
    cout << "Caracteres reales: 36\n\n";

    for (int longitud = 3; longitud <= 10; longitud++) {
        cout << "Longitud " << longitud << ": "
             << potencia(BASE, longitud)
             << " combinaciones\n";
    }

    cout << "\n37^10 solicitado en el enunciado: "
         << potencia(37, 10) << "\n";

    cout << "36^10 correcto para A-Z y 0-9: "
         << potencia(36, 10) << "\n";

    cout << "\n[ACLARACION] A-Z contiene 26 caracteres y\n";
    cout << "0-9 contiene 10. En total son 36, no 37.\n";

    cout << "[AVISO] Agotar el espacio de longitud 10\n";
    cout << "puede requerir mucho mas que unos minutos.\n";
}

void pruebasAutomaticas() {
    titulo("PRUEBAS VALIDAS E INVALIDAS");

    struct Caso {
        string clave;
        int minimo;
        int maximo;
        bool esperado;
    };

    Caso casos[] = {
        {"ABC", 3, 3, true},
        {"A0Z", 3, 3, true},
        {"AAAAAAA", 7, 10, true},
        {"AAAAAAAAAA", 7, 10, true},
        {"AB", 3, 3, false},
        {"ABCD", 3, 3, false},
        {"abc", 3, 3, false},
        {"AB@", 3, 3, false},
        {"", 3, 3, false},
        {"AAAAAA", 7, 10, false},
        {"AAAAAAAAAAA", 7, 10, false},
        {"AAAA AAA", 7, 10, false}
    };

    int aprobadas = 0;
    int total = sizeof(casos) / sizeof(casos[0]);

    for (int i = 0; i < total; i++) {
        bool obtenido = validarClave(
            casos[i].clave,
            casos[i].minimo,
            casos[i].maximo
        );

        bool correcto = obtenido == casos[i].esperado;

        cout << "[PRUEBA " << i + 1 << "] Clave: \""
             << casos[i].clave << "\" | "
             << (correcto ? "APROBADA" : "FALLIDA")
             << "\n";

        if (correcto) aprobadas++;
    }

    unsigned long long indice = 0;
    bool conversion = claveAIndice("ABC", indice);
    char reconstruida[11] = {};

    if (conversion) indiceAClave(indice, 3, reconstruida);

    bool reversible = conversion &&
                      string(reconstruida) == "ABC";

    cout << "[PRUEBA INDICE] "
         << (reversible ? "APROBADA" : "FALLIDA") << "\n";

    total++;
    if (reversible) aprobadas++;

    cout << "\nResultado: " << aprobadas
         << "/" << total << " pruebas aprobadas.\n";

    cout << "[INFO] Estas son pruebas de validacion y\n";
    cout << "conversion. Las pruebas GPU se realizan\n";
    cout << "con las opciones 3 y 4 del menu.\n";
}

void mostrarMenu() {
    titulo("SISTEMA DE BUSQUEDA PARALELA CUDA");

    cout << "1. Mostrar configuracion GPU y CUDA\n";
    cout << "2. Calcular espacio de busqueda\n";
    cout << "3. Buscar clave de 3 caracteres\n";
    cout << "4. Buscar clave de 7 a 10 caracteres\n";
    cout << "5. Ejecutar pruebas validas e invalidas\n";
    cout << "6. Mostrar resumen de resultados\n";
    cout << "7. Salir\n";

    separador();
}

int main() {
    cout << "INTEGRANTES: " << INTEGRANTES << "\n";

    int ejecuciones = 0;
    int encontradas = 0;
    int noEncontradas = 0;
    int errores = 0;
    double tiempoAcumulado = 0.0;

    bool salir = false;

    while (!salir) {
        mostrarMenu();

        int opcion = static_cast<int>(
            pedirNumero("Seleccione una opcion: ", 1, 7)
        );

        switch (opcion) {
            case 1:
                mostrarGPU();
                break;

            case 2:
                mostrarEspacio();
                break;

            case 3:
            case 4: {
                int minimo = opcion == 3 ? 3 : 7;
                int maximo = opcion == 3 ? 3 : 10;

                titulo("INTRODUCIR CLAVE DE PRUEBA");

                cout << "Solo se permiten A-Z y 0-9.\n";
                cout << "Longitud permitida: "
                     << minimo << " a " << maximo << "\n";

                if (opcion == 4) {
                    cout << "Ejemplo rapido: AAAAAA0\n";
                    cout << "Ejemplo de 10: AAAAAAAAA0\n";
                }

                cout << "Introduzca la clave: ";
                string clave;

                if (!getline(cin, clave)) {
                    cout << "[ERROR] No se pudo leer la clave.\n";
                    errores++;
                    break;
                }

                if (!validarClave(clave, minimo, maximo)) {
                    cout << "[ERROR] Clave invalida.\n";
                    cout << "Revise longitud y caracteres permitidos.\n";
                    errores++;
                    break;
                }

                unsigned long long espacio =
                    potencia(BASE, static_cast<int>(clave.size()));

                unsigned long long inicio = 0;
                unsigned long long presupuesto = espacio;

                if (opcion == 4) {
                    inicio = pedirNumero(
                        "Indice inicial (0 para comenzar): ",
                        0,
                        espacio - 1
                    );

                    presupuesto = pedirNumero(
                        "Maximo de indices a revisar (1 a 50000000): ",
                        1,
                        LIMITE_MAXIMO
                    );
                }

                Resultado r = ejecutarBusqueda(
                    clave, inicio, presupuesto
                );

                mostrarResultado(clave, r);

                ejecuciones++;
                tiempoAcumulado += r.tiempoTotal;

                if (r.error) errores++;
                else if (r.encontrado) encontradas++;
                else noEncontradas++;

                break;
            }

            case 5:
                pruebasAutomaticas();
                break;

            case 6:
                titulo("RESUMEN GENERAL");

                cout << "Ejecuciones GPU: " << ejecuciones << "\n";
                cout << "Claves encontradas: " << encontradas << "\n";
                cout << "Busquedas sin hallazgo: "
                     << noEncontradas << "\n";
                cout << "Errores: " << errores << "\n";
                cout << fixed << setprecision(6);
                cout << "Tiempo acumulado: "
                     << tiempoAcumulado << " segundos\n";
                cout << "Configuracion: "
                     << BLOQUES << " bloques x "
                     << HILOS_POR_BLOQUE << " hilos\n";
                break;

            case 7:
                titulo("FINALIZANDO PROGRAMA");
                cout << "Proceso finalizado correctamente.\n";
                salir = true;
                break;
        }
    }

    cout << "INTEGRANTES: " << INTEGRANTES << "\n";
    return 0;
}


Writing busqueda_fuerza_bruta.cu


In [3]:

!nvcc -std=c++17 -O2 busqueda_fuerza_bruta.cu -o busqueda_fuerza_bruta


busqueda_fuerza_bruta.cu(32): warning #177-D: variable "LIMITE_PREDETERMINADO" was declared but never referenced
  const unsigned long long LIMITE_PREDETERMINADO = 8192000ULL;
                           ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"



In [5]:

!./busqueda_fuerza_bruta


INTEGRANTES: Guzman Lopez Paola Nicole | Mondragon Gambino Juan Sinuhe | Morales Limon Jennifer Nataly

 SISTEMA DE BUSQUEDA PARALELA CUDA
1. Mostrar configuracion GPU y CUDA
2. Calcular espacio de busqueda
3. Buscar clave de 3 caracteres
4. Buscar clave de 7 a 10 caracteres
5. Ejecutar pruebas validas e invalidas
6. Mostrar resumen de resultados
7. Salir
Seleccione una opcion: 1

 INFORMACION DE GPU
Dispositivo: 0
Nombre: Tesla T4
Multiprocesadores: 40
Hilos maximos por bloque: 1024
Memoria global (MB): 14912
CUDA Compute Capability: 7.5
Bloques configurados: 1000
Hilos por bloque: 1024
Total de hilos: 1024000

 SISTEMA DE BUSQUEDA PARALELA CUDA
1. Mostrar configuracion GPU y CUDA
2. Calcular espacio de busqueda
3. Buscar clave de 3 caracteres
4. Buscar clave de 7 a 10 caracteres
5. Ejecutar pruebas validas e invalidas
6. Mostrar resumen de resultados
7. Salir
Seleccione una opcion: 2

 ESPACIO DE BUSQUEDA
Alfabeto: ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789
Caracteres reales: 36

Longitud 